In [ ]:
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,852 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,929 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,669 kB]
Get:

In [ ]:
import threading
import subprocess

# Start Ollama service in background

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [ ]:
!ollama pull granite4:350m

In [ ]:
!uv pip install ollama

Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 445ms
Prepared 1 package in 20ms
Installed 1 package in 2ms
 + ollama==0.6.1


In [ ]:
!uv pip install rich

Using Python 3.12.12 environment at: /usr
Audited 1 package in 114ms


In [ ]:
import json
from rich import print
from ollama import chat
model = 'granite4:350m'

In [ ]:
import sqlite3
from datetime import datetime

DATABASE_NAME = 'biblioteca.db'

In [ ]:
def get_db_connection():
    """Establishes a connection to the SQLite database."""
    conn = sqlite3.connect(DATABASE_NAME)
    conn.row_factory = sqlite3.Row # Allows accessing columns by name
    return conn

In [ ]:
def create_table():
    """Creates the products table if it doesn't already exist."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS livros (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nome TEXT NOT NULL,
            editora TEXT NOT NULL,
            autora TEXT NOT NULL,
            genero TEXT NOT NULL
        )
    ''')
    conn.commit()
    conn.close()
    print("Tabela criada com sucesso!")

In [ ]:
create_table()

Tabela criada com sucesso!

In [ ]:
def create_livro(nome, editora, autora, genero) -> str:
    conn = get_db_connection()
    cursor = conn.cursor()
    try:
        cursor.execute('''
            INSERT INTO livros (nome, editora, autora, genero)
            VALUES (?, ?, ?, ?)
        ''', (nome, editora, autora, genero))
        conn.commit()
    except sqlite3.IntegrityError:
        print(f"Erro")
    conn.close()

In [ ]:
create_livro('crime e castigo','editora 35','dostoievski','romance')
create_livro('craque neto vs neyma', 'editora 36', 'emrao', 'curiosidade')
create_livro('menor', 'editora 35', 'teste', 'teste')

In [ ]:
def todos_livros() -> str:
    """
    busque por todos os livros
    """

    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM livros")
    livros = cursor.fetchall()
    conn.close()
    return '\n'.join([', '.join(map(str, row)) for row in livros])

In [ ]:
todos_livros()

'1, crime e castigo, editora 35, dostoievski, romance\n2, crime e castigo, editora 35, dostoievski, romance\n3, craque neto vs neyma, editora 36, emrao, curiosidade\n4, crime e castigo, editora 35, dostoievski, romance\n5, craque neto vs neyma, editora 36, emrao, curiosidade\n6, menor, editora 35, teste, teste'

In [ ]:
def buscar_por_nome(nome) -> str:
    """
    Lista todos os livros na ordem natural do banco de dados (por ID).
    NÃO USE esta função se o usuário pedir qualquer tipo de ordenação ou filtro por tamanho.
    """

    conn = get_db_connection()
    cursor = conn.cursor()

    query = "SELECT * FROM livros WHERE nome LIKE ?"
    cursor.execute(query, ('%' + nome + '%',))

    livros = cursor.fetchall()
    conn.close()

    if not livros:
        return "Nenhum livro encontrado."

    resultado = []
    for livro in livros:
        id_, nome_livro, editora, autora, genero = livro
        resultado.append(f"ID: {id_} Nome: {nome_livro} Editora: {editora} Autora: {autora} Gênero: {genero}")

    return (resultado)

In [ ]:
buscar_por_nome('crime e castigo')

['ID: 1 Nome: crime e castigo Editora: editora 35 Autora: dostoievski Gênero: romance',
 'ID: 2 Nome: crime e castigo Editora: editora 35 Autora: dostoievski Gênero: romance',
 'ID: 4 Nome: crime e castigo Editora: editora 35 Autora: dostoievski Gênero: romance']

In [ ]:
def ordenar_por_tamanho_nome() -> str:
    """
    ESTA É A ÚNICA FUNÇÃO QUE ORDENA. Use exclusivamente para pedidos de
    organizar por TAMANHO, COMPRIMENTO ou QUANTIDADE DE CARACTERES do nome.
    """

    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM livros")
    livros = cursor.fetchall()
    conn.close()

    if not livros:
        return "Nenhum livro encontrado no banco de dados."

    # Ordena pelo tamanho da string no índice 1 (nome).
    livros_ordenados = sorted(livros, key=lambda x: len(str(x[1] or "")))

    resultado = []
    for livro in livros_ordenados:
        id_, nome_livro, editora, autora, genero = livro
        resultado.append(f"ID: {id_} | Nome: {nome_livro} | Autora: {autora}")

    return '\n'.join(resultado)

# --- Fluxo de Chat Corrigido ---

messages = [{'role': 'user', 'content': 'ordene todos os livros por tamanho do nome'}]
print('Prompt:', messages[0]['content'])

# Chamada do modelo
response = chat(model, messages=messages, tools=[buscar_por_nome, todos_livros, ordenar_por_tamanho_nome])

if response.message.tool_calls:
    tool = response.message.tool_calls[0]
    nome_funcao = tool.function.name
    argumentos = tool.function.arguments

    print(f'Função chamada: {nome_funcao}({argumentos})')

    if nome_funcao in locals():
        funcao_para_chamar = locals()[nome_funcao]

        # Resolve o erro de TypeError:
        # Tenta chamar com argumentos; se falhar porque a IA inventou argumentos vazios, chama sem nada.
        try:
            if argumentos:
                result = funcao_para_chamar(**argumentos)
            else:
                result = funcao_para_chamar()
        except TypeError:
            result = funcao_para_chamar()

        print(f'--- Resultado ---\n{result}')
    else:
        print(f"Erro: Função {nome_funcao} não encontrada.")
else:
    print('Resposta da IA:', response.message.content)

Prompt: ordene todos os livros por tamanho do nome

Função chamada: todos_livros({'': {}})

--- Resultado ---
1, crime e castigo, editora 35, dostoievski, romance
2, crime e castigo, editora 35, dostoievski, romance
3, craque neto vs neyma, editora 36, emrao, curiosidade
4, crime e castigo, editora 35, dostoievski, romance
5, craque neto vs neyma, editora 36, emrao, curiosidade
6, menor, editora 35, teste, teste

In [ ]:
messages = [{'role': 'user', 'content': 'busque o livro com nome crime e castigo'}]
print('Prompt:', messages[0]['content'])

response = chat(model, messages=messages, tools=[buscar_por_nome, todos_livros])
###print(response.message.tool_calls) ### na segunda vez ele fica none - nulo
if response.message.tool_calls:
  tool = response.message.tool_calls[0]
  print(f'Funcao chamada: {tool.function.name}({tool.function.arguments})')

  result = locals()[tool.function.name](**tool.function.arguments) ### chamando uma func() usando seu nome em string
  print(f'Result: {result}')

  ###messages.append(response.message)
  ###messages.append({'role': 'tool', 'content': result})

  ###print('Messages:', messages)

  ###final = chat(model, messages=messages) ### isso eh uma grande instabilidade - usar o LM para gerar a resposta
  ###print('Response:', final.message.content)
else:
  print('Response:', response.message.content)

Prompt: busque o livro com nome crime e castigo

Funcao chamada: buscar_por_nome({'nome': 'crime'})

Result: ['ID: 1 Nome: crime e castigo Editora: editora 35 Autora: dostoievski Gênero: romance']